# Country Missingness Scoring

Identifies countries systematically absent from data they should have.
Classifies each (country, year) as dissolved, microstate, failed, degraded, reporting, or strong.

**Phase 1: Model Definition**

In [1]:
using Revise
using InteractiveUtils

includet("phase1/functions/load_phase1.jl")

╔══════════════════════════════════════════════════════════════════════════╗
║ QoG METADATA JOINING - PHASE 0 LOADED                                ║
╚══════════════════════════════════════════════════════════════════════════╝

Quick Start:
    metadata = join_metadata()              # Run full pipeline (single isomorphism check)
    metadata = join_metadata_with_cascade()  # Run cascade (strictest → loosest), then union on slug
    quick_check()                             # Diagnostic check
    inspect_exceptions()                      # Review configuration
    show_usage()                              # Detailed documentation

Pipeline Steps:
    1. ingest_and_normalize()            # Load & normalize sources (PDF = qog_slugs_temporal.csv; min_year/max_year ingested)
    2. align_id_variables!(...)          # Harmonize ID vars
    3. run_isomorphism_cascade(...)      # Strictest → loosest until success; returns (stata_df, pdf_df, arrow_df) for union on slug
    4. unify_and_join(..

In [4]:
using CSV, DataFrames

df = load_augmented_or_build()
meta_df = CSV.read("data/qog_metadata_plus2.csv", DataFrame)
println("Loaded: $(nrow(df)) rows, $(nrow(meta_df)) slugs")

✓ Checksum verified: data/qog_std_ts_jan25_aug.arrow
✓ Loaded: 12391 rows × 2014 cols from data/qog_std_ts_jan25_aug.arrow
  ggis_rowid unique: ✓ | Required columns: ✓ | Missing regions: 0 ✓
Loaded: 12391 rows, 2010 slugs


## Run Pipeline

In [5]:
result = run_country_missingness(df, meta_df)

Step 0 — Country Temporal Profiles
    Total countries: 200
    Active: 194
    Dissolved: 6
      CSK Czechoslovakia (dissolved 1992, last data 1992)
      DDR German Democratic Republic (dissolved 1990, last data 1990)
      YMD Yemen Democratic (dissolved 1990, last data 1989)
      SUN USSR (dissolved 1991, last data 1991)
      YUG Yugoslavia (dissolved 1992, last data 1991)
      XTI Tibet (dissolved 1959, last data 1950)
    Microstates: 177
      AFG Afghanistan (pop 41K)
      ALB Albania (pop 3K)
      DZA Algeria (pop 46K)
      AND Andorra (pop 0K)
      AGO Angola (pop 36K)
      ATG Antigua and Barbuda (pop 0K)
      AZE Azerbaijan (pop 10K)
      ARG Argentina (pop 45K)
      AUS Australia (pop 26K)
      AUT Austria (pop 9K)
      BHS Bahamas (the) (pop 0K)
      BHR Bahrain (pop 2K)
      ARM Armenia (pop 4K)
      BRB Barbados (pop 0K)
      BEL Belgium (pop 12K)
      BTN Bhutan (pop 1K)
      BOL Bolivia (Plurinational State of) (pop 12K)
      BIH Bosnia and Herzeg

(profiles = 200×13 DataFrame
 Row │ ident_ccode  ident_ccodealp  ident_cname                        country ⋯
     │ Int64        String          String                             Int64   ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │           4  AFG             Afghanistan                                ⋯
   2 │           8  ALB             Albania
   3 │          12  DZA             Algeria
   4 │          20  AND             Andorra
   5 │          24  AGO             Angola                                     ⋯
   6 │          28  ATG             Antigua and Barbuda
   7 │          31  AZE             Azerbaijan
   8 │          32  ARG             Argentina
   9 │          36  AUS             Australia                                  ⋯
  10 │          40  AUT             Austria
  11 │          44  BHS             Bahamas (the)
  ⋮  │      ⋮             ⋮                         ⋮                          ⋱
 191 │         840  USA      

## Country Profiles

Dissolved states, microstates, and temporal spans.

In [ ]:
# Dissolved states
filter(r -> r.is_dissolved, result.profiles)

In [ ]:
# Microstates
filter(r -> r.is_microstate, result.profiles)

## Status Distribution

How many country-years fall in each category?

In [ ]:
sort(combine(groupby(result.status, :country_status), nrow => :count), :count, rev=true)

## Failed & Degraded Countries

Which active countries are losing data coverage?

In [ ]:
# Countries ever classified as failed (excluding dissolved/micro)
failed = filter(r -> r.country_status == "failed", result.status)
failed_countries = unique(failed.ident_ccode)
println("Countries with 'failed' years: $(length(failed_countries))")

# Show their trajectory: status by decade
for ccode in failed_countries[1:min(10, length(failed_countries))]
    rows = filter(r -> r.ident_ccode == ccode, result.status)
    alpha = filter(r -> r.ident_ccode == ccode, result.profiles).ident_ccodealp[1]
    name = filter(r -> r.ident_ccode == ccode, result.profiles).ident_cname[1]
    println("\n  $alpha $name:")
    for decade_start in [1990, 2000, 2010, 2020]
        decade = filter(r -> decade_start <= r.ident_year < decade_start + 10, rows)
        if nrow(decade) > 0
            statuses = unique(decade.country_status)
            avg_cov = round(mean(decade.global_coverage_pct) * 100, digits=1)
            println("    $(decade_start)s: $(join(statuses, "/")) ($(avg_cov)% avg coverage)")
        end
    end
end

## Revised Slug Penetration

Slugs that gain penetration when failed states are excluded from the denominator.

In [ ]:
# Top gainers
first(result.penetration, 20)

In [ ]:
# How many slugs cross the 95% threshold with revised denominator?
original_global = count(r -> r.original_penetration >= 0.95, eachrow(result.penetration))
revised_global = count(r -> r.revised_penetration >= 0.95, eachrow(result.penetration))
println("Slugs ≥95% penetration:")
println("  Original denominator: $original_global")
println("  Revised denominator:  $revised_global")
println("  New globals:          $(revised_global - original_global)")

## Save Flags

In [ ]:
# CSV.write("data/country_missingness_flags.csv", result.flags)
# println("\u2705 Saved country_missingness_flags.csv")